<a href="https://colab.research.google.com/github/suheylozdemir/Credit-Repayment-Prediction/blob/main/Legalmind_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Kütüphaneleri Yükle
!pip install langchain langchain-community langchain-google-genai langchain_openai langchain_anthropic chromadb sentence-transformers torch openai anthropic -q
!pip install --upgrade --quiet pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.3/222.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 6.8 MB/s eta 0:00:

In [ ]:
# @title Kütüphaneleri Import Et

import os
import json
import shutil
import torch
import getpass
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.schema import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from google.colab import userdata

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# @title Dosyaları Yükleme
# Burası .txt, .pdf ve .doc uzantılı dosyaları yüklemeye yarıyor.
# .doc kısmı tam çalışmıyor!!!

def load_documents_multiple_formats(directory_path, processed_files):
    documents = []
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        if filename not in processed_files:
            print(f"Processing document: {filename}")
            if filename.endswith(".txt"):
                loader = TextLoader(file_path)
                documents.extend(loader.load())
            elif filename.endswith(".doc"):
                loader = UnstructuredWordDocumentLoader(file_path)
                documents.extend(loader.load())
            elif filename.endswith(".pdf"):
                loader = PyPDFLoader(file_path)
                documents.extend(loader.load())
            elif filename.endswith(".csv"):
                 loader = CSVLoader(file_path)
                 documents.extend(loader.load())
        else:
            print(f"Skipping already processed document: {filename}")
    return documents

In [ ]:
# @title İşlenmiş Dosyaları Kaydedip Yükleme
# Daha önceden işlenmiş bir dosyayı tekrar işlememek için gerekli fonksiyonlar.

def load_processed_files(file_path="processed_files.json"):
    try:
        with open(file_path, 'r') as f:
            return set(json.load(f))
    except FileNotFoundError:
        return set()

def save_processed_files(processed_files, file_path="processed_files.json"):
    with open(file_path, 'w') as f:
        json.dump(list(processed_files), f)

In [ ]:
# @title Dokümanları Parçalara Ayırma
# Burası dokümanları küçük parçalara ayırma işini yapıyor.
# Şuan kullanılan chunk size yani parça boyutu 500, chunk overlap yani bir öncekinden ne kadar alacağı 100.
# Eğer sonuç istediğimiz gibi olmazsa bu chunk_size ve chunk_overlap'i değiştirip tekrar denememiz lazım.

def chunk_documents(documents, chunk_size=500, chunk_overlap=100):
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
  chunks = text_splitter.split_documents(documents)
  return chunks

In [ ]:
# @title Embedding İşlemi İçin İlgili Modelin Oluşturulması
# Burası her parçayı sayısal bir vektöre dönüştürmek için kullanılacak embedding modelini oluşturuyor.
# Kullandığım model "all-mpnet-base-v2". Başka modeller de denenebilir.
# İşlem hızı açısından GPU ile çalışmak daha iyi sonuç vereceği için eğer GPU varsa GPU, yoksa CPU kullanıyor.

device = "cuda" if torch.cuda.is_available() else "cpu"

def create_embeddings():
  embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2", model_kwargs={'device': device})
  return embeddings

In [ ]:
# @title Vector Veritabanının Oluşturulması
# Burada vektör veritabanı oluşturulup içine embedding işlemi yapılan parçalar vektör olarak ekleniyor.
# Vektör veritabanı olarak Chroma DB kullandım (açık kaynak). Başka veritabanları da denenebilir.

def create_vector_database(chunks, embeddings, persist_directory):
    vectordb = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=persist_directory)
    return vectordb

In [ ]:
# @title Varolan Vector Veritabanının Yüklenmesi
# Burası varolan vektör veritabanını yükler.
# Bu şu yüzden önemli, embedding işlemini yapmak çok uzun sürdüğü için, veritabanına kaydedip tekrar tekrar kullanabiliriz.

def load_vector_database(persist_directory, embeddings):
  vectordb = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
  return vectordb

In [ ]:
# @title LLM'in Eklenmesi
# Burada sorulan soruya ilgili doküman yardımıyla cevap verecek LLM ekleniyor.
# Ben denemeler bedava olsun diye Google'ın "Gemini-1.5-Flash" modelini kullandım. Başka modeller de denenebilir.

def initialize_llm():
    llm_id = input("""Hangi LLM'i kullanmak istersin (modelin yanındaki sayıyı girin)?
(1) Gemini 1.5 Flash
(2) Claude 3 Sonnet
(3) Deepseek v3
""")

    match llm_id:
        case '1':
            print('Gemini 1.5 Flash modeli seçildi.')
            os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
            llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
        case '2':
            print('Claude 3 Sonnet modeli seçildi.')
            os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
            llm = ChatAnthropic(model_name="claude-3-sonnet-20240229")
        case '3':
            print('Deepseek v3 modeli seçildi.')
            os.environ["OPENAI_API_KEY"] = userdata.get('DEEPSEEK_API_KEY')
            llm = ChatOpenAI(model_name="deepseek-chat", openai_api_base='https://api.deepseek.com')
        case _:
            print("Geçersiz seçim. Varsayılan olarak Deepseek v3 modeli seçildi.")
            os.environ["OPENAI_API_KEY"] = userdata.get('DEEPSEEK_API_KEY')
            llm = ChatOpenAI(model_name="deepseek-chat", openai_api_base='https://api.deepseek.com')

    return llm

In [ ]:
# @title RAG Chain Fonksiyonun Oluşturulması
# Burası vektör veritabanı ve LLM'i birleştiren bir RAG Chain fonksiyonu oluşturur.
# Burada aynı zamanda sisteme cevabını şöyle ver gibi bir komut da verebiliriz.

def create_rag_chain(vectordb, llm, search_k):
    template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Always answer in Turkish language.

{context}

Question: {question}
Answer:
    """

    QA_PROMPT = PromptTemplate(template=template, input_variables=["context", "question"])
    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectordb.as_retriever(search_kwargs={'k': search_k}), chain_type_kwargs={"prompt": QA_PROMPT})

    return qa

In [ ]:
# @title RAG Sistemine Soru Gönderip Alma

def query_rag_system(qa_chain, query):
    result = qa_chain.run(query)
    return result

In [ ]:
# @title RAG Sistemine Gönderilen Parçalar Hakkında Bilgi Alma
# Burası ilgili parçaların benzerlik puanı ve kaynak dokümanı hakkında bilgi verir.

def retrieve_relevant_chunks(vectordb, query, search_k):
    retriever = vectordb.as_retriever(search_kwargs={'k': search_k})
    retrieved_docs = retriever.get_relevant_documents(query)

    # Benzerlik puanını hesapla
    results = vectordb.similarity_search_with_relevance_scores(query, k=search_k)

    # Metadata'yı hesaplanan puanla güncelle
    for i, doc in enumerate(retrieved_docs):
        if i < len(results):
            doc.metadata['score'] = results[i][1]
    return retrieved_docs

In [ ]:
# @title Dokümanları Vektör Veritabanına Yükleme
# Burada "input_documents" klasöründeki tüm dokümanlar yüklenir, parçalara ayrılır, embedding işlemi yapılır ve vektör veritabanına yüklenir.
# Dokümanın işlemi başarılı bir şekilde bittikten sonra "processed_documents" klasörüne taşınır ve "processed_files.json" dosyasına eklenir.
# Bu sayede eğer bir doküman işlenirken hata oluşursa, kod yeniden çalıştırıldığında önceden işlenmiş dosyalar tekrardan işlenmez.
# Yeni dokümanlar geldikçe "input_documents" kısmına eklenerek sürece dahil edilirler.

input_dir = '/content/input_documents'
archive_dir = '/content/processed_documents'
persist_directory = 'db'
processed_files_file = "processed_files.json"

processed_files = load_processed_files(processed_files_file)
embeddings = create_embeddings()

# Vektör veritabanı önceden oluşturulmuşsa varolan yüklenir, yoksa sıfırdan oluşturulur.
if os.path.exists(persist_directory):
    vectordb = load_vector_database(persist_directory, embeddings)
    print("Loaded Existing Vector Database!")
else:
    vectordb = None
    print("Created Vector Database!")


# Dokümanların işlenmesi
if os.path.exists(input_dir):
    for filename in os.listdir(input_dir):
        file_path = os.path.join(input_dir, filename)

        if filename not in processed_files:
            try:
                documents = load_documents_multiple_formats(input_dir, processed_files)
                if not documents:
                  continue
                chunks = chunk_documents(documents)
                if vectordb is None:
                    vectordb = create_vector_database(chunks, embeddings, persist_directory)
                else:
                     vectordb.add_documents(chunks)

                if not os.path.exists(archive_dir):
                    os.makedirs(archive_dir)
                shutil.move(file_path, os.path.join(archive_dir, filename))
                processed_files.add(filename)
                save_processed_files(processed_files, processed_files_file)
                print(f"Successfully processed {filename}")
            except Exception as e:
                print(f"Error processing {filename}: {e}")
        else:
            print(f"Skipping already processed document: {filename}")
else:
    print("No documents to process")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Created Vector Database!
Processing document: dosya_2.pdf
Processing document: dosya_1.pdf
Processing document: dosya_4.pdf
Processing document: dosya_5.pdf
Processing document: dosya_3.pdf
Successfully processed dosya_2.pdf
Processing document: dosya_1.pdf
Processing document: dosya_4.pdf
Processing document: dosya_5.pdf
Processing document: dosya_3.pdf
Successfully processed dosya_1.pdf
Processing document: dosya_4.pdf
Processing document: dosya_5.pdf
Processing document: dosya_3.pdf
Successfully processed dosya_4.pdf
Processing document: dosya_5.pdf
Processing document: dosya_3.pdf
Successfully processed dosya_5.pdf
Processing document: dosya_3.pdf
Successfully processed dosya_3.pdf


In [15]:
# @title Soru - Cevap
# Burada RAG sistemi kullanılarak sorulara cevap verilir.
# Örnek soru: Haciz işlemlerinin durdurulması konusunda bilgi verir misin?

search_k = 3 # Bu en alakalı kaç parçanın LLM'e göndereceğini gösteriyor. İstenirse bu sayı değiştirilebilir.

llm = initialize_llm()

while True:
    query = input("Soru: ")
    if query.lower() == "exit":
        break

    retrieved_docs = retrieve_relevant_chunks(vectordb, query, search_k)
    print(f"\n--- Gönderilen {len(retrieved_docs)} Parça ---")

    for i, doc in enumerate(retrieved_docs):
        score_str = "N/A"
        if 'score' in doc.metadata:
            score = doc.metadata['score']
            score_str = f"{score * 100:.2f}%"

        print(f"Parça {i+1}:")
        print(f"  Kaynak Doküman: {doc.metadata['source']}")
        print(f"  Benzerlik Puanı: {score_str}")
        print(f"  İçerik: {doc.page_content}...")

    qa = create_rag_chain(vectordb, llm, search_k)

    answer = query_rag_system(qa, query)
    print("\nCevap:", answer)

Claude 3 Sonnet modeli seçildi.

--- Gönderilen 3 Parça ---
Parça 1:
  Kaynak Doküman: /content/input_documents/dosya_1.pdf
  Benzerlik Puanı: 78.82%
  İçerik: uyuşmazlığın  çözümü,haklı  neden  ve  kamu  yararının  bulunması  halinde  kanunkoyucu
tarafından  adli  yargıya  bırakılabilir.  İtirazkonusu  kural,  trafik  kazasında  zarar  görenin...
Parça 2:
  Kaynak Doküman: /content/input_documents/dosya_1.pdf
  Benzerlik Puanı: 78.82%
  İçerik: uyuşmazlığın  çözümü,haklı  neden  ve  kamu  yararının  bulunması  halinde  kanunkoyucu
tarafından  adli  yargıya  bırakılabilir.  İtirazkonusu  kural,  trafik  kazasında  zarar  görenin...
Parça 3:
  Kaynak Doküman: /content/input_documents/dosya_5.pdf
  Benzerlik Puanı: 75.68%
  İçerik: adli yargının görevlendirilmesikonusunda kanun koyucunun mutlak bir takdir yetkisinin
bulunduğunusöylemek  olanaklı  değildir.  Ancak,  idari  yargınındenetimine  bağlı  olması
gereken idari biruyuşmazlığın çözümü, haklı nedenve kamu yararının bulunması halind

KeyboardInterrupt: Interrupted by user

In [ ]:
# @title Debugging
# 1. Yüklenen Dokümanları Bastır (parçalara ayırma işlemi olmadan önce)
print("Loaded Documents:")
for doc in documents:
  print(doc.page_content)

# 2. Parçaları Bastır (LLM çalışmadan önce)
retriever = vectordb.as_retriever()
retrieved_docs = retriever.get_relevant_documents(query)
print("\nRetrieved Chunks:")
i = 1
for doc in retrieved_docs:
    print(f"Chunk {i}:")
    i += 1
    print(doc.page_content)

# 3. LLM'in Oluşturduğu Cevabı Bastır
print("\nLLM Generated Text:")
print(answer)